# 06 — Asking a real agent fleet a plain-English question

Every notebook so far ran everything inside this one kernel process. This
one is different: it launches **real, separate `scarlet-agents` containers**
(one head, two workers) using the Docker SDK, waits for them to come online,
then asks a question in plain English - the head's LLM decides which
`scarlet_agentic_harness` skills to call (here: `sum`, twice, with different
transforms) and reports back a natural-language answer, having actually
dispatched to and coordinated with the two worker containers over Redis.

Each worker's private data comes from its own CSV file (`data/worker1.csv`,
`data/worker2.csv`) - real separate containers, so each just gets its own
`CSV_PATH` env var; no in-process trickery needed to keep them from stepping
on each other's data, unlike a single-kernel simulation would.

This notebook needs the Docker socket mounted in (see
`examples/notebook-agents/docker-compose.yml`) - it controls sibling
containers on the host, a real step up from anything the other five
notebooks do.

In [ ]:
import os, time, json
import docker

APP_ID = "tutorial_stats"
HEAD_NAME = "tutorial-agent-head"
WORKER_NAMES = ["tutorial-agent-worker-1", "tutorial-agent-worker-2"]
ALL_NAMES = [HEAD_NAME] + WORKER_NAMES

client = docker.from_env()

## Cleanup

Fixed container names mean a previous run's containers (if this notebook
crashed, or you're just re-running it) need to be removed before starting
fresh - otherwise launching a container with the same name fails outright.

In [ ]:
def cleanup():
    for name in ALL_NAMES:
        try:
            client.containers.get(name).remove(force=True)
        except docker.errors.NotFound:
            pass

cleanup()
print("cleaned up")

## Provide your LLM credentials

These are only ever passed into the **head** container's environment - the
notebook's own process never calls the LLM directly, and the workers never
see these values at all (they don't need an LLM backend to run `sum`).

In [ ]:
import getpass

LLM_BASE_URL = input("LLM base URL (OpenAI-compatible, e.g. an Anthropic-compatible endpoint): ").strip()
LLM_API_KEY = getpass.getpass("LLM API key (hidden): ").strip()
LLM_MODEL = input("LLM model name: ").strip()

## Find this container's own network and host path

Containers launched via the Docker SDK from inside this notebook don't
automatically join the same network `docker compose` put this container on
- that has to be passed explicitly. Rather than hardcode the compose
project's network name (fragile - it depends on the directory name docker
compose was run from), inspect this container's own attributes and reuse
whatever network it's already on.

CSV bind-mounts have a similar wrinkle: a bind-mount source path is
resolved by the Docker **daemon**, not by this container - so the path has
to be the real path on the host, not `/data/...` as seen from inside this
container. `docker-compose.yml` passes that host path in as
`HOST_PROJECT_DIR` for exactly this reason (`docker compose`'s `${PWD}`
interpolation, evaluated on the host, not in any container).

In [ ]:
own_container = client.containers.get(os.environ["HOSTNAME"])
network_name = next(iter(own_container.attrs["NetworkSettings"]["Networks"]))
print(f"joining new containers to network: {network_name}")

host_project_dir = os.environ["HOST_PROJECT_DIR"]
host_data_dir = f"{host_project_dir}/data"

## Build the worker image

Workers read their private data via `CsvConnector`, which needs `duckdb` -
not installed in the published `scarlet-agents` image (it's installed
`--no-deps` there on purpose, see `harness/Dockerfile`). Rather than
change that shared image for one tutorial, build a small tutorial-only
image on top of it, from a Dockerfile already baked into *this* image at
`/worker-image` (see `docker/jupyter-agents/worker-image/Dockerfile`).

In [ ]:
TUTORIAL_IMAGE_TAG = "tutorial-scarlet-agent:latest"

image, build_logs = client.images.build(
    path="/worker-image",
    tag=TUTORIAL_IMAGE_TAG,
    buildargs={"AGENTS_VERSION": "latest"},
)
print(f"built {TUTORIAL_IMAGE_TAG}")

## Bring up the fleet

All three containers run this same built image, not the raw published
`scarlet-agents:latest` - see `docker/jupyter-agents/worker-image/Dockerfile`'s
own comment for why the head needs this too (a real, pre-existing bug in
the published image, unrelated to CSV support). The head runs
`mcp_server.py` explicitly - `scarlet-agents`' own default `CMD`
(supervisord -> `__main__.py`) never exposes MCP at all (see
`harness/Dockerfile`'s own comment on this). The workers use the image's
default command unchanged - a plain worker needs nothing beyond its env
vars, `ROLE=worker` is the default even.

In [ ]:
common_env = {
    "APP_ID": APP_ID,
    "REDIS_HOST": "redis",
    "REDIS_PORT": "6379",
    "REDIS_AUTH_TOKEN": os.environ.get("REDIS_AUTH_TOKEN", ""),
}

head = client.containers.run(
    TUTORIAL_IMAGE_TAG,
    name=HEAD_NAME,
    command=["python", "-m", "scarlet_agentic_harness.mcp_server"],
    environment={
        **common_env,
        "ROLE": "head",
        "NODE_ADDRESS": "head",
        "LLM_BASE_URL": LLM_BASE_URL,
        "LLM_API_KEY": LLM_API_KEY,
        "LLM_MODEL": LLM_MODEL,
        "MCP_TRANSPORT": "streamable-http",
        "MCP_HOST": "0.0.0.0",
        "MCP_PORT": "8090",
    },
    network=network_name,
    detach=True,
)

for i, worker_name in enumerate(WORKER_NAMES, start=1):
    client.containers.run(
        TUTORIAL_IMAGE_TAG,
        name=worker_name,
        environment={
            **common_env,
            "ROLE": "worker",
            "NODE_ADDRESS": f"worker{i}",
            "CSV_PATH": f"/data/worker{i}.csv",
        },
        volumes={f"{host_data_dir}/worker{i}.csv": {"bind": f"/data/worker{i}.csv", "mode": "ro"}},
        network=network_name,
        detach=True,
    )

print("launched:", ", ".join(ALL_NAMES))

## Wait for the fleet to come online

Rather than a fixed sleep, poll `GatherStatus()` on the head bus - the same
Messenger primitive notebook 01 used - until both workers have registered.
`scarlets` is already available in this image (it's the base `scarlet-agents`
extends), so this is a real readiness check, not a guess.

Every scarlet-agentic-harness process registers under `f"{app_id}_{node_address}"`
(`HarnessConfig.agent_id`) - matching the `NODE_ADDRESS` values just set
above, the two workers' expected ids are computed directly rather than
guessed.

In [ ]:
from scarlets.messaging import Messenger

os.environ.setdefault("REDIS_HOST", "redis")
os.environ.setdefault("REDIS_PORT", "6379")

expected_worker_ids = {f"{APP_ID}_worker1", f"{APP_ID}_worker2"}
status_checker = Messenger(f"{APP_ID}_headagent", agentId="notebook_status_check")

deadline = time.time() + 30
online = set()
while time.time() < deadline and not expected_worker_ids.issubset(online):
    online = set(status_checker.GatherStatus().keys())
    if not expected_worker_ids.issubset(online):
        time.sleep(1)

print(f"online: {sorted(online & expected_worker_ids)}")

## Ask a plain-English question

The head decides, on its own, to call `sum` twice (`transform=identity` for
Sigma(x), `transform=square` for Sigma(x^2)) across both workers' real CSV
data, then composes mean/variance/stddev itself before replying - no
distributed "variance" skill exists, by design (see `sum.py`'s own
docstring).

In [ ]:
import asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def ask_scarlet_agent(message: str) -> str:
    url = f"http://{HEAD_NAME}:8090/mcp"
    async with streamable_http_client(url) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool("ask_scarlet_agent", {"message": message})
            return result.content[0].text

answer = asyncio.run(ask_scarlet_agent(
    "What are the sum, mean, and variance of the readings held across all "
    "currently online workers?"
))
print(answer)

## Cleanup (teardown)

These are real containers - stop and remove them so this notebook doesn't
leave anything running on the host.

In [ ]:
cleanup()
print("cleaned up")